# Phase 1 Dataset Smell Test

This notebook is the **gate deliverable** for Phase 1. Each section answers one question that, if the answer looks wrong, blocks modeling work in Phase 2.

Run this end-to-end after a full ingest (`eonet ingest --catalogs eonet,usgs,noaa,firms --since 2000-01-01`). Inspect the output by eye.


In [ ]:
import polars as pl

from eonet_cascades.config import DataConfig
from eonet_cascades.data.store import EventStore

cfg = DataConfig()
store = EventStore(cfg.duckdb_path)
store.init_schema()
df = store.query_events()
print(f"Total events: {df.height:,}")
print()
print("Per catalog:")
print(df.group_by("source_catalog").len().sort("len", descending=True))
print()
print("Per mark:")
print(df.group_by("mark").len().sort("len", descending=True))


In [ ]:
import matplotlib.pyplot as plt

yearly = (
    df.with_columns(pl.col("time_start").dt.year().alias("year"))
      .group_by(["year", "source_catalog"]).len()
      .sort("year")
)
fig, ax = plt.subplots(figsize=(10, 5))
for cat, group in yearly.partition_by("source_catalog", as_dict=True).items():
    ax.plot(group["year"], group["len"], label=cat, marker="o")
ax.set_yscale("log")
ax.set_xlabel("Year")
ax.set_ylabel("Events (log)")
ax.set_title("Event counts per year, per catalog")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
wildfires = df.filter(pl.col("mark") == "wildfire")
sample = wildfires.sample(min(20000, wildfires.height), seed=0) if wildfires.height > 0 else wildfires
ax.scatter(sample["longitude"], sample["latitude"], s=2, alpha=0.3, c="orangered")
ax.set_xlim(-130, -65)
ax.set_ylim(14, 50)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Wildfires (n={sample.height:,}) — spatial coverage check")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Dedup health.
# Note: under the current dedup implementation every event has a dedup_group_id.
# Multi-event clusters are what matter — group by id and inspect sizes.
groups = df.group_by("dedup_group_id").len()
multi = groups.filter(pl.col("len") > 1)
n_total = df.height
n_clustered = df.join(multi.select("dedup_group_id"), on="dedup_group_id", how="inner").height
print(f"Events in multi-event clusters: {n_clustered:,} / {n_total:,} ({100*n_clustered/max(1,n_total):.2f}%)")
print(f"Number of multi-event clusters: {multi.height:,}")
print(f"Mean cluster size: {multi['len'].mean() if multi.height else 0:.2f}")
print(f"Max cluster size: {multi['len'].max() if multi.height else 0}")
print()
print("Largest clusters (sanity check — should be hurricanes / floods, not earthquakes):")
top = multi.sort("len", descending=True).head(10)
for row in top.iter_rows(named=True):
    gid = row["dedup_group_id"]
    members = df.filter(pl.col("dedup_group_id") == gid)
    marks = members["mark"].unique().to_list()
    cats = members["source_catalog"].unique().to_list()
    t0 = members["time_start"].min()
    print(f"  {gid}  size={row['len']:>4}  marks={marks}  catalogs={cats}  t0={t0}")


## Phase 1 Gate Checklist

Before moving to Phase 2, confirm by eye:

- [ ] Per-catalog counts are non-zero and within an order of magnitude of expectations (USGS in the hundreds of thousands; FIRMS in the millions; NOAA in the hundreds of thousands; EONET in the tens of thousands).
- [ ] Per-mark counts cover all 12 unified marks; no mark has zero events.
- [ ] The yearly time series shows reasonable coverage from 2000 onward; no suspicious gaps.
- [ ] The CONUS wildfire map shows points concentrated in expected regions (California, Pacific Northwest, Southwest) — not uniform random.
- [ ] Dedup health: multi-event clusters are dominated by tropical cyclones, severe storms, floods — **not** earthquakes (tight thresholds should keep quake-clustering near 0%). If quakes are over-clustering, revisit thresholds.

If any item fails, fix the data layer before opening Plan 2.
